# Procesamiento de texto — rama NLP

**Itaca SmartDiag** · Camilo Cabrera

Convierte la columna de texto libre `respuesta_texto` en secuencias de
enteros de longitud fija, listas para alimentar una capa `Embedding`.

El vocabulario se aprende de las respuestas de entrenamiento y se serializa
a disco, de modo que el servicio de inferencia pueda reconstruir la capa sin
depender de este notebook.

| | |
|---|---|
| Entrada | `splits/train.csv`, `val.csv`, `test.csv` |
| Salida | `artifacts/vocab.txt` |
| Salida | `output/X_text_train.npy`, `X_text_val.npy`, `X_text_test.npy` |

Los splits provienen del preprocesamiento tabular y se tratan como solo
lectura.

## Configuración

In [1]:
import os

import numpy as np
import pandas as pd
import tensorflow as tf

In [2]:
SPLITS_DIR = "splits"
ARTIFACTS_DIR = "artifacts"
OUTPUT_DIR = "output"

TRAIN_PATH = f"{SPLITS_DIR}/train.csv"
VAL_PATH = f"{SPLITS_DIR}/val.csv"
TEST_PATH = f"{SPLITS_DIR}/test.csv"

In [3]:
VOCAB_PATH = f"{ARTIFACTS_DIR}/vocab.txt"

X_TEXT_TRAIN_PATH = f"{OUTPUT_DIR}/X_text_train.npy"
X_TEXT_VAL_PATH = f"{OUTPUT_DIR}/X_text_val.npy"
X_TEXT_TEST_PATH = f"{OUTPUT_DIR}/X_text_test.npy"

In [4]:
TEXT_COL = "respuesta_texto"

MAX_TOKENS = 150
SEQUENCE_LENGTH = 16
STANDARDIZE = "lower_and_strip_punctuation"

RANDOM_STATE = 42

In [5]:
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)

for d in (ARTIFACTS_DIR, OUTPUT_DIR):
    os.makedirs(d, exist_ok=True)

In [6]:
print(f"TensorFlow: {tf.__version__}")
print(f"Keras: {tf.keras.__version__}")
print(f"MAX_TOKENS={MAX_TOKENS} | SEQUENCE_LENGTH={SEQUENCE_LENGTH}")
print(f"standardize={STANDARDIZE}")

TensorFlow: 2.21.0
Keras: 3.15.0
MAX_TOKENS=150 | SEQUENCE_LENGTH=16
standardize=lower_and_strip_punctuation


## Carga de los splits

In [7]:
train_df = pd.read_csv(TRAIN_PATH, encoding="utf-8")
val_df = pd.read_csv(VAL_PATH, encoding="utf-8")
test_df = pd.read_csv(TEST_PATH, encoding="utf-8")

SPLITS = [("train", train_df), ("val", val_df), ("test", test_df)]

In [8]:
for name, frame in SPLITS:
    n_unicos = frame[TEXT_COL].nunique()
    print(f"{name:>5}: {len(frame):>4} filas | textos unicos: {n_unicos}")

total = len(train_df) + len(val_df) + len(test_df)
print(f"\nTotal: {total} filas")

train: 2250 filas | textos unicos: 32
  val:  450 filas | textos unicos: 31
 test:  300 filas | textos unicos: 30

Total: 3000 filas


Reviso la columna de texto antes de vectorizar. Un nulo pasaría a ser
el string `"nan"` al aplicar `astype(str)` y entraría al vocabulario como una
palabra más.

In [9]:
for name, frame in SPLITS:
    assert TEXT_COL in frame.columns, f"Falta la columna {TEXT_COL} en {name}"
    assert frame[TEXT_COL].notna().all(), f"Hay nulos en {TEXT_COL} en {name}"

print("Columna de texto presente y sin nulos en los tres splits: OK")

Columna de texto presente y sin nulos en los tres splits: OK


## Conteo de filas por partición

Tomo el número de filas de los propios CSV en lugar de fijarlo como
constante, para que las validaciones de shape sigan siendo válidas si el
reparto cambia más adelante. El reparto implementado en el preprocesamiento
tabular no coincide con el que suponen otros documentos del proyecto, así que
dejo la comparación visible.

In [10]:
EXPECTED_ROWS = {
    "train": len(train_df),
    "val": len(val_df),
    "test": len(test_df),
}

SPEC_ROWS = {"train": 2100, "val": 450, "test": 450}

In [11]:
print("Comparacion documentada vs splits reales:")
print(f"{'split':>6} | {'doc':>6} | {'real':>6} | coincide")

for k in ("train", "val", "test"):
    ok = "si" if SPEC_ROWS[k] == EXPECTED_ROWS[k] else "NO"
    print(f"{k:>6} | {SPEC_ROWS[k]:>6} | {EXPECTED_ROWS[k]:>6} | {ok:>8}")

Comparacion documentada vs splits reales:
 split |    doc |   real | coincide
 train |   2100 |   2250 |       NO
   val |    450 |    450 |       si
  test |    450 |    300 |       NO


In [12]:
reparto = " / ".join(
    f"{EXPECTED_ROWS[k] / total:.0%}" for k in ("train", "val", "test")
)
print(f"Reparto real: {reparto}")

Reparto real: 75% / 15% / 10%


## Capa de vectorización

`SEQUENCE_LENGTH = 16` cubre la respuesta más larga del corpus sin
truncarla. `MAX_TOKENS = 150` deja margen sobre las ~106 palabras distintas
que aporta el corpus. `lower_and_strip_punctuation` baja a minúsculas y
elimina signos, pero no descompone caracteres Unicode, así que las tildes
sobreviven.

In [13]:
vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
    standardize=STANDARDIZE,
)

In [14]:
print("Capa TextVectorization creada (aun sin adaptar).")
print(f"  max_tokens             = {MAX_TOKENS}")
print(f"  output_mode            = int")
print(f"  output_sequence_length = {SEQUENCE_LENGTH}")
print(f"  standardize            = {STANDARDIZE}")

Capa TextVectorization creada (aun sin adaptar).
  max_tokens             = 150
  output_mode            = int
  output_sequence_length = 16
  standardize            = lower_and_strip_punctuation


## Aprendizaje del vocabulario

Aprendo el vocabulario únicamente con las respuestas de entrenamiento.
Si val o test participaran, sus palabras quedarían indexadas y las secuencias
de evaluación cargarían información que el modelo no vio al entrenar.

In [15]:
train_texts = train_df[TEXT_COL].astype(str).values

print(f"Textos usados en adapt(): {len(train_texts)}")
print(f"Textos de val NO usados:  {len(val_df)}")
print(f"Textos de test NO usados: {len(test_df)}")

Textos usados en adapt(): 2250
Textos de val NO usados:  450
Textos de test NO usados: 300


In [16]:
vectorizer.adapt(train_texts)

print("Capa adaptada. Vocabulario aprendido solo con train.")

Capa adaptada. Vocabulario aprendido solo con train.


Compruebo el aislamiento midiendo el resultado y no la intención:
extraigo las palabras que solo existen en val o test y verifico que ninguna
haya llegado al vocabulario.

In [17]:
vocab_check = set(str(t) for t in vectorizer.get_vocabulary())


def tokens_de(frame):
    serie = frame[TEXT_COL].astype(str).str.lower()
    serie = serie.str.replace(r"[^\w\s\u00C0-\u017F]", "", regex=True)
    return set(w for fila in serie for w in fila.split())

In [18]:
en_evaluacion = tokens_de(val_df) | tokens_de(test_df)
solo_val_test = en_evaluacion - tokens_de(train_df)
fugados = solo_val_test & vocab_check

print(f"Tokens que aparecen solo en val/test: {len(solo_val_test)}")
print(f"De esos, cuantos entraron al vocabulario: {len(fugados)}")

Tokens que aparecen solo en val/test: 0
De esos, cuantos entraron al vocabulario: 0


In [19]:
assert not fugados, f"Leakage: tokens de val/test en vocab -> {fugados}"

print("Sin leakage: ningun token exclusivo de val/test en el vocabulario")

Sin leakage: ningun token exclusivo de val/test en el vocabulario


## Estructura del vocabulario

In [20]:
vocab = vectorizer.get_vocabulary()

print(f"Tamano real del vocabulario: {len(vocab)} tokens")
print(f"Limite configurado (MAX_TOKENS): {MAX_TOKENS}")
print(f"Margen sin usar: {MAX_TOKENS - len(vocab)} posiciones")

Tamano real del vocabulario: 108 tokens
Limite configurado (MAX_TOKENS): 150
Margen sin usar: 42 posiciones


In [21]:
print("Primeros 20 tokens (los mas frecuentes):")

for i, token in enumerate(vocab[:20]):
    etiqueta = str(token) if str(token) != "" else "(cadena vacia)"
    print(f"  [{i:>3}] {etiqueta}")

Primeros 20 tokens (los mas frecuentes):
  [  0] (cadena vacia)
  [  1] [UNK]
  [  2] de
  [  3] se
  [  4] todo
  [  5] pero
  [  6] nos
  [  7] tenemos
  [  8] no
  [  9] la
  [ 10] procesos
  [ 11] mejorar
  [ 12] hay
  [ 13] en
  [ 14] pronto
  [ 15] gustaría
  [ 16] y
  [ 17] usamos
  [ 18] el
  [ 19] es


In [22]:
print("Ultimos 5 tokens (los menos frecuentes):")

for i, token in enumerate(vocab[-5:], start=len(vocab) - 5):
    print(f"  [{i:>3}] {token}")

Ultimos 5 tokens (los menos frecuentes):
  [103] guía
  [104] decisión
  [105] cada
  [106] autónomo
  [107] analítica


Keras antepone dos posiciones reservadas al vocabulario ordenado por
frecuencia: el índice 0 rellena las secuencias cortas y el 1 recoge cualquier
palabra desconocida.

In [23]:
print("Posiciones reservadas del vocabulario:")
print(f"  indice 0 -> {repr(str(vocab[0]))}  (padding, cadena vacia)")
print(f"  indice 1 -> {repr(str(vocab[1]))}  (OOV / desconocido)")

Posiciones reservadas del vocabulario:
  indice 0 -> ''  (padding, cadena vacia)
  indice 1 -> '[UNK]'  (OOV / desconocido)


In [24]:
assert str(vocab[0]) == "", "El indice 0 deberia ser el token de padding"
assert str(vocab[1]) == "[UNK]", "El indice 1 deberia ser [UNK]"

palabras_reales = len(vocab) - 2
print(f"Palabras reales aprendidas: {palabras_reales}")
print(f"Total con reservados: {palabras_reales} + 2 = {len(vocab)}")

Palabras reales aprendidas: 106
Total con reservados: 106 + 2 = 108


## Tratamiento de acentos

Verifico cómo quedaron las palabras acentuadas. Una normalización
Unicode habría colapsado `empirico` y `empírico` en un mismo índice, y el
vocabulario dejaría de coincidir con el texto que llega del formulario.

In [25]:
acentos = "áéíóúñüÁÉÍÓÚÑÜ"

con_tilde = sorted(
    str(t) for t in vocab
    if any(a in str(t) for a in acentos)
)

print(f"Tokens con tilde o enye: {len(con_tilde)} de {len(vocab)}")
print(f"Lista completa: {con_tilde}")

Tokens con tilde o enye: 21 de 108
Lista completa: ['analítica', 'autónomo', 'básicas', 'básico', 'cálculo', 'decisión', 'documentación', 'empírico', 'está', 'estándares', 'gustaría', 'guía', 'guías', 'información', 'integración', 'mayoría', 'metodologías', 'tecnológicas', 'todavía', 'ágiles', 'áreas']


In [26]:
vocab_str = [str(t) for t in vocab]

for palabra in ["documentación", "empírico"]:
    presente = palabra in vocab_str
    indice = vocab_str.index(palabra) if presente else None
    estado = f"presente en el indice {indice}" if presente else "AUSENTE"
    print(f"  '{palabra}': {estado}")

  'documentación': presente en el indice 73
  'empírico': presente en el indice 72


In [27]:
for sin_tilde in ["documentacion", "empirico"]:
    assert sin_tilde not in vocab_str, (
        f"Aparecio '{sin_tilde}' sin tilde: se estan alterando acentos"
    )

print("Observacion: las tildes se conservan intactas. No hay duplicados")
print("acentuado/sin acentuar, asi que el vocabulario queda alineado con")
print("el texto real que llegara desde el formulario en produccion.")

Observacion: las tildes se conservan intactas. No hay duplicados
acentuado/sin acentuar, asi que el vocabulario queda alineado con
el texto real que llegara desde el formulario en produccion.


## Secuencias generadas

Contrasto el texto original contra la secuencia producida: las
posiciones ocupadas corresponden a las palabras y el resto se rellena con
ceros hasta completar la longitud fija.

In [28]:
EJEMPLOS = train_df[TEXT_COL].astype(str).head(5).tolist()

secuencias_ejemplo = vectorizer(np.array(EJEMPLOS)).numpy()

In [29]:
for i, (texto, seq) in enumerate(zip(EJEMPLOS, secuencias_ejemplo), 1):
    n_palabras = len(texto.split())
    n_padding = int((seq == 0).sum())

    print(f"--- Ejemplo {i} ---")
    print(f"  Original ({n_palabras} palabras): {texto}")
    print(f"  Secuencia: {seq.tolist()}")
    print(f"  Longitud: {len(seq)} (esperado {SEQUENCE_LENGTH})")
    print(f"  Ceros de padding al final: {n_padding}")
    print()

    assert len(seq) == SEQUENCE_LENGTH, "La secuencia no tiene longitud 16"

--- Ejemplo 1 ---
  Original (12 palabras): Intentamos usar software básico pero el equipo no se adapta del todo.
  Secuencia: [40, 38, 39, 42, 5, 18, 27, 8, 3, 43, 41, 4, 0, 0, 0, 0]
  Longitud: 16 (esperado 16)
  Ceros de padding al final: 4

--- Ejemplo 2 ---
  Original (9 palabras): Perdemos mucha información porque todo se anota en cuadernos.
  Secuencia: [75, 76, 77, 74, 4, 3, 79, 13, 78, 0, 0, 0, 0, 0, 0, 0]
  Longitud: 16 (esperado 16)
  Ceros de padding al final: 7

--- Ejemplo 3 ---
  Original (16 palabras): Usamos un ERP pero nos falta integración con la parte de ventas. Nos gustaría mejorar pronto.
  Secuencia: [17, 45, 48, 5, 6, 22, 47, 25, 9, 46, 2, 44, 6, 15, 11, 14]
  Longitud: 16 (esperado 16)
  Ceros de padding al final: 0

--- Ejemplo 4 ---
  Original (11 palabras): Todo es manual, dependemos de una sola persona para la contabilidad.
  Secuencia: [4, 19, 87, 88, 2, 84, 85, 86, 29, 9, 89, 0, 0, 0, 0, 0]
  Longitud: 16 (esperado 16)
  Ceros de padding al final: 5

--- 

In [30]:
print(f"Los {len(EJEMPLOS)} ejemplos tienen longitud exactamente "
      f"{SEQUENCE_LENGTH}: OK")

Los 5 ejemplos tienen longitud exactamente 16: OK


## Léxico fuera de vocabulario

Pruebo con palabras ajenas al corpus para ver el camino de palabra
desconocida, que es el que se va a dar en producción cuando un cliente
escriba algo que el vocabulario no cubre.

In [31]:
TEXTO_INVENTADO = "Implementamos blockchain y metaverso con criptomonedas"

seq_inventado = vectorizer(np.array([TEXTO_INVENTADO])).numpy()[0]
palabras = TEXTO_INVENTADO.lower().split()

print(f"Texto inventado: {TEXTO_INVENTADO}")
print(f"Secuencia: {seq_inventado.tolist()}")

Texto inventado: Implementamos blockchain y metaverso con criptomonedas
Secuencia: [1, 1, 16, 1, 25, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [32]:
print(f"{'palabra':>16} | {'indice':>6} | estado")

for palabra, indice in zip(palabras, seq_inventado):
    if indice == 1:
        estado = "[UNK]"
    elif indice > 1:
        estado = "conocida"
    else:
        estado = "padding"
    print(f"{palabra:>16} | {indice:>6} | {estado}")

         palabra | indice | estado
   implementamos |      1 | [UNK]
      blockchain |      1 | [UNK]
               y |     16 | conocida
       metaverso |      1 | [UNK]
             con |     25 | conocida
   criptomonedas |      1 | [UNK]


In [33]:
n_unk = int((seq_inventado == 1).sum())
print(f"Tokens mapeados a [UNK] = 1: {n_unk}")

assert n_unk > 0, "Se esperaba al menos un token [UNK]"
print("Prueba de vocabulario desconocido: OK")

Tokens mapeados a [UNK] = 1: 4
Prueba de vocabulario desconocido: OK


## Vectorización de los splits

La capa ya está adaptada; sobre los tres splits solo se invoca. En
ningún momento se vuelve a llamar `adapt()`. `int32` basta porque los índices
no pasan de 150.

In [34]:
def vectorizar(frame):
    textos = frame[TEXT_COL].astype(str).values
    return vectorizer(np.array(textos)).numpy().astype("int32")


X_text_train = vectorizar(train_df)
X_text_val = vectorizar(val_df)
X_text_test = vectorizar(test_df)

In [35]:
MATRICES = [
    ("X_text_train", X_text_train),
    ("X_text_val", X_text_val),
    ("X_text_test", X_text_test),
]

for nombre, matriz in MATRICES:
    print(f"{nombre:>13}: shape {matriz.shape} | dtype {matriz.dtype}")

 X_text_train: shape (2250, 16) | dtype int32
   X_text_val: shape (450, 16) | dtype int32
  X_text_test: shape (300, 16) | dtype int32


Valido shape, tipo y rango. Un índice fuera de rango rompería la capa
`Embedding` durante el entrenamiento, con un error difícil de rastrear hasta
aquí.

In [36]:
matrices = {
    "train": (X_text_train, EXPECTED_ROWS["train"]),
    "val": (X_text_val, EXPECTED_ROWS["val"]),
    "test": (X_text_test, EXPECTED_ROWS["test"]),
}

In [37]:
for nombre, (matriz, filas) in matrices.items():
    esperado = (filas, SEQUENCE_LENGTH)

    assert matriz.shape == esperado, f"{nombre}: shape {matriz.shape}"
    assert np.issubdtype(matriz.dtype, np.integer), f"{nombre}: dtype"
    assert matriz.min() >= 0, f"{nombre}: hay indices negativos"
    assert matriz.max() < len(vocab), f"{nombre}: indice fuera de rango"

In [38]:
print("Shapes validadas contra los splits reales: OK")
print("Dtype entero (int32) en las tres matrices: OK")
print(f"Rango de indices dentro de [0, {len(vocab) - 1}]: OK")

Shapes validadas contra los splits reales: OK
Dtype entero (int32) en las tres matrices: OK
Rango de indices dentro de [0, 107]: OK


In [39]:
for nombre, matriz in [("val", X_text_val), ("test", X_text_test)]:
    print(f"Tokens [UNK] en {nombre}: {int((matriz == 1).sum())}")

Tokens [UNK] en val: 0
Tokens [UNK] en test: 0


## Persistencia de las matrices

Recargo cada archivo tras escribirlo para confirmar la serialización.

In [40]:
np.save(X_TEXT_TRAIN_PATH, X_text_train)
np.save(X_TEXT_VAL_PATH, X_text_val)
np.save(X_TEXT_TEST_PATH, X_text_test)

In [41]:
RUTAS_NPY = (X_TEXT_TRAIN_PATH, X_TEXT_VAL_PATH, X_TEXT_TEST_PATH)

for ruta in RUTAS_NPY:
    recargado = np.load(ruta)
    print(f"Guardado: {ruta}")
    print(f"  shape {recargado.shape} | dtype {recargado.dtype}")

Guardado: output/X_text_train.npy


  shape (2250, 16) | dtype int32
Guardado: output/X_text_val.npy
  shape (450, 16) | dtype int32
Guardado: output/X_text_test.npy
  shape (300, 16) | dtype int32


## Serialización del vocabulario

Un token por línea, en el mismo orden que devuelve la capa. Los índices
grabados en las matrices corresponden a la posición de cada palabra en este
archivo, así que reordenarlo invalidaría tanto las matrices como cualquier
modelo entrenado con ellas. La primera línea sale vacía porque el token de
padding es la cadena vacía.

In [42]:
with open(VOCAB_PATH, "w", encoding="utf-8") as f:
    for token in vocab:
        f.write(f"{str(token)}\n")

print(f"Guardado: {VOCAB_PATH} ({len(vocab)} lineas)")

Guardado: artifacts/vocab.txt (108 lineas)


In [43]:
with open(VOCAB_PATH, encoding="utf-8") as f:
    primeras = [next(f).rstrip("\n") for _ in range(4)]

print("Primeras 4 lineas del archivo:")
for i, linea in enumerate(primeras):
    print(f"  linea {i}: {repr(linea)}")

Primeras 4 lineas del archivo:
  linea 0: ''
  linea 1: '[UNK]'
  linea 2: 'de'
  linea 3: 'se'


## Reconstrucción de la capa

Reproduzco el arranque del servicio de inferencia: una capa nueva, con
los mismos parámetros de construcción, cargada desde el archivo.

In [44]:
with open(VOCAB_PATH, encoding="utf-8") as f:
    vocab_desde_archivo = [line.rstrip("\n") for line in f]

rebuilt = tf.keras.layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
    standardize=STANDARDIZE,
)
rebuilt.set_vocabulary(vocab_desde_archivo)

In [45]:
vocab_rebuilt = [str(t) for t in rebuilt.get_vocabulary()]

print(f"Vocabulario leido del archivo: {len(vocab_desde_archivo)} tokens")
print(f"Vocabulario reconstruido: {len(vocab_rebuilt)} tokens")
print(f"Orden identico al original: {vocab_rebuilt == vocab_str}")

Vocabulario leido del archivo: 108 tokens
Vocabulario reconstruido: 108 tokens
Orden identico al original: True


In [46]:
salida_original = vectorizer(np.array(EJEMPLOS)).numpy()
salida_reconstruida = rebuilt(np.array(EJEMPLOS)).numpy()

iguales = np.array_equal(salida_original, salida_reconstruida)

In [47]:
print("Comparacion original vs reconstruida sobre los 5 ejemplos:")

for i in range(len(EJEMPLOS)):
    coincide = np.array_equal(salida_original[i], salida_reconstruida[i])
    print(f"  Ejemplo {i + 1}: {'identico' if coincide else 'DIFERENTE'}")

print(f"numpy.array_equal sobre las 5 secuencias: {iguales}")
assert iguales, "La capa reconstruida no reproduce la salida original"

Comparacion original vs reconstruida sobre los 5 ejemplos:
  Ejemplo 1: identico
  Ejemplo 2: identico
  Ejemplo 3: identico
  Ejemplo 4: identico
  Ejemplo 5: identico
numpy.array_equal sobre las 5 secuencias: True


Extiendo la comparación a los tres splits completos: cinco filas pueden
coincidir por compartir vocabulario frecuente, y así se cubren también las
palabras de cola.

In [48]:
COMPLETOS = [
    ("train", train_df, X_text_train),
    ("val", val_df, X_text_val),
    ("test", test_df, X_text_test),
]

for nombre, frame, matriz in COMPLETOS:
    textos = frame[TEXT_COL].astype(str).values
    recalculado = rebuilt(np.array(textos)).numpy().astype("int32")

    assert np.array_equal(recalculado, matriz), f"Diferencia en {nombre}"
    print(f"Split {nombre} completo reproducido de forma identica: OK")

Split train completo reproducido de forma identica: OK
Split val completo reproducido de forma identica: OK
Split test completo reproducido de forma identica: OK


In [49]:
print("Reconstruccion exacta verificada.")

Reconstruccion exacta verificada.


## Verificación final

In [50]:
resultados = []

c1_shapes = all(
    matriz.shape == (filas, SEQUENCE_LENGTH)
    for matriz, filas in matrices.values()
)
c1_dtype = all(
    np.issubdtype(m.dtype, np.integer) for m, _ in matrices.values()
)
shapes_txt = f"{X_text_train.shape} {X_text_val.shape} {X_text_test.shape}"

In [51]:
resultados.append((
    "Shapes (filas del split x 16) y dtype entero",
    c1_shapes and c1_dtype,
    f"{shapes_txt} int32 -- filas segun los splits reales",
))

In [52]:
with open(VOCAB_PATH, encoding="utf-8") as f:
    lineas = f.read().split("\n")

c2 = os.path.exists(VOCAB_PATH)
c2 = c2 and lineas[0] == "" and lineas[1] == "[UNK]"

resultados.append((
    "vocab.txt UTF-8, linea 0 vacia y linea 1 = [UNK]",
    c2,
    f"linea 0 = {repr(lineas[0])}, linea 1 = {repr(lineas[1])}",
))

In [53]:
resultados.append((
    "Vocabulario adaptado solo con train",
    len(fugados) == 0,
    f"adapt() recibio {len(train_texts)} textos; 0 tokens de val/test",
))

resultados.append((
    "Reconstruccion de la capa con igualdad exacta",
    iguales,
    "array_equal en los 5 ejemplos y en los 3 splits completos",
))

resultados.append((
    "Texto inventado produce tokens [UNK] = 1",
    n_unk > 0,
    f"{n_unk} de {len(palabras)} palabras mapeadas a [UNK]",
))

In [54]:
print("VERIFICACION FINAL")
print("=" * 68)

for descripcion, ok, detalle in resultados:
    print(f"[{'OK' if ok else 'FALLA'}] {descripcion}")
    print(f"       {detalle}")

print("=" * 68)
assert all(ok for _, ok, _ in resultados), "Hay invariantes sin cumplir"
print("Las 5 invariantes se cumplen.")

VERIFICACION FINAL
[OK] Shapes (filas del split x 16) y dtype entero
       (2250, 16) (450, 16) (300, 16) int32 -- filas segun los splits reales
[OK] vocab.txt UTF-8, linea 0 vacia y linea 1 = [UNK]
       linea 0 = '', linea 1 = '[UNK]'
[OK] Vocabulario adaptado solo con train
       adapt() recibio 2250 textos; 0 tokens de val/test
[OK] Reconstruccion de la capa con igualdad exacta
       array_equal en los 5 ejemplos y en los 3 splits completos
[OK] Texto inventado produce tokens [UNK] = 1
       4 de 6 palabras mapeadas a [UNK]
Las 5 invariantes se cumplen.


## Artefactos generados

In [55]:
ARTEFACTOS = (VOCAB_PATH,) + RUTAS_NPY

print("ARTEFACTOS")
print("=" * 68)

for ruta in ARTEFACTOS:
    print(f"  {ruta:<32} {os.path.getsize(ruta):>9,} bytes")

print("=" * 68)

ARTEFACTOS
  artifacts/vocab.txt                    833 bytes
  output/X_text_train.npy            144,128 bytes
  output/X_text_val.npy               28,928 bytes
  output/X_text_test.npy              19,328 bytes


In [56]:
filas_totales = len(X_text_train) + len(X_text_val) + len(X_text_test)

print(f"Vocabulario: {len(vocab)} tokens "
      f"({len(vocab) - 2} palabras + 2 reservados)")
print(f"Longitud de secuencia: {SEQUENCE_LENGTH}")
print(f"Filas vectorizadas: {filas_totales}")

Vocabulario: 108 tokens (106 palabras + 2 reservados)
Longitud de secuencia: 16
Filas vectorizadas: 3000
